# Levels 01–03 — geometry, relighting, and spatial gesture control

Run the cells in order in a **local Jupyter or VS Code notebook** with the
project Python environment selected. The live camera opens an OpenCV window.
Set `LIVE_BACKEND="colab"` only if using the optional Google Colab path. The synthetic checks need no
camera or model download. The photo cell is optional; the webcam cell does
not depend on a photo upload. Outside Colab, set `PHOTO_PATH` for a photo;
local Level 3 uses MediaPipe hand tracking, with OpenCV sliders as fallback.

Level 3 adds local and optional browser hand tracking for smooth XYZ light control, with
calibration, tracking feedback, and manual fallback. See the live instructions.

Level 2 adds a virtual point light with Lambertian diffuse and Blinn–Phong
specular shading. Run the lighting helper and its synthetic checks before the
photo or live cells. The notebook is self-contained; no repository modules
are imported.

This notebook estimates **relative geometry, not distance in meters**.
Depth Anything V2 Small predicts relative inverse depth. We retain its
floating-point output and map it to a positive inverse-depth proxy before
taking its reciprocal. This creates an approximate surface in arbitrary
units; neither its shape nor scale is a calibrated reconstruction.

The assumed horizontal field of view is adjustable. Supply calibrated
`fx`, `fy`, `cx`, `cy` at the processed image resolution when available.
Calibration alone does not remove the depth model's scale/shift ambiguity.

The default live output shows relighting only; optional diagnostic panels
use the **same captured frame**. Resolution and backend settings are in the setup cell. The displayed rate measures
completed Python capture/inference/render-update cycles, including transfers
when using Colab; it does not measure browser paint time. It excludes the first
three warmup cycles. No real-time performance is claimed until measured on
the demo machine.

In [1]:
# Optional dependency setup. Keep your CUDA-compatible torch/torchvision build.
# MediaPipe needs contrib OpenCV; install only one cv2 provider. Restart kernel afterwards.
%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python-headless
%pip install torch torchvision "numpy>=1.24,<2" matplotlib pillow "opencv-contrib-python>=4.8,<4.12" "transformers>=4.40,<5" "mediapipe==0.10.32"


Note: you may need to restart the kernel to use updated packages.


  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  Preparing metadata (pyproject.toml) did not run successfully.
  exit code: 1
  
  [21 lines of output]
  + C:\Users\USER\vision_hackathon\nrw-geometry-engine\.venv\Scripts\python.exe C:\Users\USER\AppData\Local\Temp\pip-install-6e9nscth\numpy_82bf501bc3e94310ad310f10db91ae20\vendored-meson\meson\meson.py setup C:\Users\USER\AppData\Local\Temp\pip-install-6e9nscth\numpy_82bf501bc3e94310ad310f10db91ae20 C:\Users\USER\AppData\Local\Temp\pip-install-6e9nscth\numpy_82bf501bc3e94310ad310f10db91ae20\.mesonpy-hbrhpk2k -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\USER\AppData\Local\Temp\pip-install-6e9nscth\numpy_82bf501bc3e94310ad310f10db91ae20\.mesonpy-hbrhpk2k\meson-python-native-file.ini
  The Meson build system
  Version: 1.2.99
  Source dir: C:\Users\USER\AppData\Local\Temp\pip-install-6e9nscth\numpy_82bf501bc3e94310ad310f10db91ae20
  Build dir: C:\Users\USER\AppData\Local\Temp\pip-install-6e9nscth\numpy_82bf501bc

In [2]:
import io
import math
import time
from collections import deque
from base64 import b64decode

import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Javascript, Markdown, Image as DisplayImage
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32
# Local notebook defaults. Re-run setup + model after changing INPUT_SIZE.
LIVE_BACKEND = "local"  # OpenCV camera/window; "colab" retains the browser experiment.
CAMERA_INDEX = 0
CAMERA_FPS = 30  # Camera request, not a guaranteed processing rate.
PERFORMANCE_PRESET = "fast"  # "fast", "balanced", or "detail"
PRESETS = {"fast": (196, 320), "balanced": (224, 480), "detail": (294, 640)}
INPUT_SIZE, CAPTURE_WIDTH = PRESETS[PERFORMANCE_PRESET]
GEOMETRY_BACKEND = "auto"  # Startup comparison chooses CPU or GPU smoothing.
# To customize further, override INPUT_SIZE (multiple of 14) and CAPTURE_WIDTH here.
JPEG_QUALITY = 75  # Colab only. Local display uses raw pixels, no JPEG transfer.
SHOW_DIAGNOSTICS = False  # True restores synchronized RGB/Z/normals/relit panels.
HAND_CONTROL = True  # Level 3. G toggles local hand/manual control.
HAND_MODEL_PATH = "models/hand_landmarker.task"  # Local asset, prepared below.
SHOW_HAND_LANDMARKS = True  # Output-only dots; never enter depth inference.
STATUS_INTERVAL = 1.0  # Seconds between text updates.
PROFILE_STAGES = True  # CUDA events; no per-stage synchronize calls.
FAST_PREPROCESS = True  # OpenCV cubic resize; reference processor remains available.
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True  # Fixed video/model shapes after warmup.
ASSUMED_HFOV_DEG = 60.0
MODEL_ID = "depth-anything/Depth-Anything-V2-Small-hf"
print(f"Device: {DEVICE}; inference dtype: {DTYPE}; input target: {INPUT_SIZE}")
print(f"Backend: {LIVE_BACKEND}; preset: {PERFORMANCE_PRESET}; capture width: {CAPTURE_WIDTH}")

C:\Users\USER\vision_hackathon\nrw-geometry-engine\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda; inference dtype: torch.float16; input target: 196
Backend: local; preset: fast; capture width: 320


In [3]:
# Self-contained geometry helpers: this cell requires no model or camera.
import math
import cv2
import numpy as np
import torch
import torch.nn.functional as F


class RelativeGeometryDepth:
    """Float inverse-depth prediction -> positive Z proxy in arbitrary units.

    Z = 1 / (inverse_floor + normalized_prediction).
    The offset/range are modeling choices, not recovered physical parameters.
    In video, an EMA stabilizes normalization bounds only. It does not align
    moving objects or provide motion-compensated temporal depth filtering.
    """

    def __init__(self, bounds_alpha=0.15, inverse_floor=0.2):
        if not 0 < bounds_alpha <= 1:
            raise ValueError("bounds_alpha must be in (0, 1].")
        if not math.isfinite(inverse_floor) or inverse_floor <= 0:
            raise ValueError("inverse_floor must be finite and positive.")
        self.alpha = bounds_alpha
        self.inverse_floor = inverse_floor
        self.reset()

    def reset(self):
        self.bounds = None
        self.shape = None

    def __call__(self, prediction, temporal=False):
        raw = np.asarray(prediction, dtype=np.float32)
        if raw.ndim != 2 or min(raw.shape) < 3 or not np.isfinite(raw).all():
            raise ValueError("Prediction must be a finite HxW array, at least 3x3.")
        lo, hi = (float(v) for v in np.percentile(raw, [2, 98]))
        if temporal and self.bounds is not None and self.shape == raw.shape:
            lo = (1 - self.alpha) * self.bounds[0] + self.alpha * lo
            hi = (1 - self.alpha) * self.bounds[1] + self.alpha * hi
        self.bounds, self.shape = (lo, hi), raw.shape
        if hi - lo < 1e-6:
            relative_inverse = np.full_like(raw, 0.5)
        else:
            relative_inverse = np.clip((raw - lo) / (hi - lo), 0, 1)
        if hasattr(cv2, "bilateralFilter"):
            smooth = cv2.bilateralFilter(
                relative_inverse, d=5, sigmaColor=0.08, sigmaSpace=3
            )
        else:
            # Some minimal cv2 builds omit imgproc; keep geometry usable.
            tensor = torch.from_numpy(relative_inverse)[None, None]
            tensor = F.pad(tensor, (1, 1, 1, 1), mode="replicate")
            smooth = F.avg_pool2d(tensor, kernel_size=3, stride=1)[0, 0].numpy()
        return (1.0 / (self.inverse_floor + smooth)).astype(np.float32)


class ScharrNormalEstimator:
    """Perspective tangents and unit normals for a positive camera-Z map.

    Coordinates: X right, Y down, Z away from camera. Tx cross Ty uses
    the positive-Z orientation on a fronto-parallel plane. For outward
    normals facing the camera in a later renderer, negate these normals.
    Returned normals are signed XYZ tensors, NOT RGB-encoded values.
    """

    def __init__(self, height, width, fx=None, fy=None, cx=None, cy=None,
                 hfov_deg=60.0, device="cpu"):
        if height < 3 or width < 3:
            raise ValueError("Image dimensions must be at least 3x3.")
        if not 0 < hfov_deg < 180:
            raise ValueError("hfov_deg must be between 0 and 180.")
        self.H, self.W = height, width
        self.device = torch.device(device)
        fx = float(fx) if fx is not None else width / (2 * math.tan(math.radians(hfov_deg) / 2))
        fy = float(fy) if fy is not None else fx  # Assumed square pixels.
        cx = float(cx) if cx is not None else (width - 1) / 2
        cy = float(cy) if cy is not None else (height - 1) / 2
        if not all(math.isfinite(v) for v in (fx, fy, cx, cy)) or min(fx, fy) <= 0:
            raise ValueError("Intrinsics must be finite; fx and fy must be positive.")
        v, u = torch.meshgrid(
            torch.arange(height, device=self.device, dtype=torch.float32),
            torch.arange(width, device=self.device, dtype=torch.float32),
            indexing="ij",
        )
        self.ray_grid = torch.stack(((u - cx) / fx, (v - cy) / fy, torch.ones_like(u)), -1)
        kx = torch.tensor(
            [[-3., 0., 3.], [-10., 0., 10.], [-3., 0., 3.]],
            device=self.device,
        ) / 32.0
        self.kx = kx[None, None].repeat(3, 1, 1, 1)
        self.ky = kx.T[None, None].repeat(3, 1, 1, 1)

    @torch.inference_mode()
    def compute_normals_and_coords(self, depth, validate=True):
        depth = torch.as_tensor(depth, device=self.device, dtype=torch.float32)
        if tuple(depth.shape) != (self.H, self.W):
            raise ValueError("Depth dimensions must match the estimator.")
        if validate and (not torch.isfinite(depth).all().item() or (depth <= 0).any().item()):
            raise ValueError("Camera-Z values must be finite and strictly positive.")
        points = self.ray_grid * depth[..., None]
        # Replicate padding avoids differentiating against fictitious zero-Z
        # neighbors; do not overwrite real border orientations with +Z.
        p = F.pad(points.permute(2, 0, 1)[None], (1, 1, 1, 1), mode="replicate")
        tx = F.conv2d(p, self.kx, groups=3)[0].permute(1, 2, 0)
        ty = F.conv2d(p, self.ky, groups=3)[0].permute(1, 2, 0)
        # Normalize tangents first so small scene units do not shrink the
        # cross product below the normalization epsilon.
        tx, ty = F.normalize(tx, dim=-1), F.normalize(ty, dim=-1)
        cross = torch.cross(tx, ty, dim=-1)
        length = torch.linalg.vector_norm(cross, dim=-1, keepdim=True)
        valid = length[..., 0] > 1e-6
        normals = cross / length.clamp_min(1e-6)
        normals = torch.where(valid[..., None], normals, torch.zeros_like(normals))
        return normals, points, valid


def normals_to_rgb(normals, valid):
    rgb = ((normals + 1) * 127.5).round().clamp(0, 255).to(torch.uint8)
    rgb = torch.where(valid[..., None], rgb, torch.zeros_like(rgb))
    return rgb.cpu().numpy()

class DeviceRelativeGeometryDepth:
    """GPU-resident Z proxy, with sampled percentile bounds and 5x5 bilateral.

    A bounded sample (at most 4096 pixels) estimates the 2/98 percentiles.
    This differs slightly from the reference NumPy/OpenCV mapper. The filter
    uses OpenCV's circular radius-2 support and reflect-101 borders.
    Bounds EMA stabilizes scale only; it does not freeze or reuse geometry.
    """
    def __init__(self, bounds_alpha=0.15, inverse_floor=0.2, max_samples=4096):
        if not 0 < bounds_alpha <= 1 or not math.isfinite(inverse_floor) or inverse_floor <= 0:
            raise ValueError("Invalid normalization settings.")
        if max_samples < 2:
            raise ValueError("max_samples must be at least 2.")
        self.alpha, self.inverse_floor = bounds_alpha, inverse_floor
        self.max_samples = max_samples
        self.reset()

    def reset(self):
        self.bounds = self.shape = self.spatial = None

    @torch.inference_mode()
    def __call__(self, prediction, temporal=False, validate=True):
        raw = torch.as_tensor(prediction).float()
        if raw.ndim != 2 or min(raw.shape) < 3:
            raise ValueError("Prediction must be HxW, at least 3x3.")
        if validate and not torch.isfinite(raw).all().item():
            raise ValueError("Prediction must be finite.")
        if self.shape != tuple(raw.shape) or self.spatial is None or self.spatial.device != raw.device:
            self.reset()
            self.shape = tuple(raw.shape)
            y, x = torch.meshgrid(torch.arange(-2, 3, device=raw.device),
                                  torch.arange(-2, 3, device=raw.device), indexing="ij")
            radius_sq = (x*x + y*y).float().reshape(1, 25, 1)
            self.spatial = torch.where(radius_sq <= 4, -radius_sq / 18,
                                       torch.full_like(radius_sq, -float("inf")))
        # Avoid copying the full prediction to NumPy just to find its bounds.
        flat = raw.flatten()
        stride = max(1, math.ceil(flat.numel() / self.max_samples))
        sample = flat[::stride].sort().values
        # Explicit linear quantiles avoid a CUDA scalar validation sync in
        # torch.quantile. The percentile ranks depend only on the static shape.
        bounds_list = []
        for q in (0.02, 0.98):
            rank = (sample.numel() - 1) * q
            low, high = math.floor(rank), math.ceil(rank)
            bounds_list.append(sample[low].lerp(sample[high], rank-low))
        bounds = torch.stack(bounds_list)
        if temporal and self.bounds is not None:
            bounds = self.bounds.lerp(bounds, self.alpha)
        self.bounds = bounds
        lo, hi = bounds.unbind()
        relative = ((raw - lo) / (hi-lo).clamp_min(1e-6)).clamp(0, 1)
        relative = torch.where(hi-lo < 1e-6, torch.full_like(relative, 0.5), relative)
        patches = F.unfold(F.pad(relative[None, None], (2, 2, 2, 2), mode="reflect"), 5)
        distance = patches - relative.reshape(1, 1, -1)
        weights = torch.softmax(self.spatial - distance.square() / (2 * 0.08**2), dim=1)
        smooth = (weights * patches).sum(1).reshape_as(raw)
        return (self.inverse_floor + smooth).reciprocal()


In [4]:
def run_synthetic_checks(device="cpu"):
    estimator = ScharrNormalEstimator(41, 61, fx=75, fy=80, device=device)
    rays = estimator.ray_grid

    # Fronto-parallel plane, including borders and different global scales.
    for z in (0.001, 1.0, 10.0):
        depth = torch.full((41, 61), z, device=estimator.device)
        normals, points, valid = estimator.compute_normals_and_coords(depth)
        expected = torch.tensor([0., 0., 1.], device=estimator.device).expand_as(normals)
        assert valid.all().item()
        torch.testing.assert_close(normals, expected, atol=2e-5, rtol=0)
        torch.testing.assert_close(points[..., 2], depth)

    # Analytic tilted plane n.P = d. Its normal must remain constant,
    # including image borders and after scaling the entire scene.
    expected = F.normalize(torch.tensor([-0.2, 0.3, 1.], device=estimator.device), dim=0)
    depth = 2.0 / (rays * expected).sum(dim=-1)
    for scale in (0.001, 1.0, 10.0):
        normals, _, valid = estimator.compute_normals_and_coords(depth * scale)
        assert valid.all().item()
        torch.testing.assert_close(normals, expected.expand_as(normals), atol=4e-4, rtol=0)
        torch.testing.assert_close(torch.linalg.vector_norm(normals, dim=-1), torch.ones_like(depth), atol=1e-6, rtol=0)

    # Invalid Z must fail explicitly rather than fabricate a surface.
    for bad in (torch.zeros((41, 61)), torch.full((41, 61), float("nan")), torch.ones((4, 4))):
        try:
            estimator.compute_normals_and_coords(bad)
        except ValueError:
            pass
        else:
            raise AssertionError("Invalid Z was accepted.")

    mapper = RelativeGeometryDepth()
    raw = np.tile(np.linspace(0, 1, 61, dtype=np.float32), (41, 1))
    proxy = mapper(raw)
    assert np.isfinite(proxy).all() and (proxy > 0).all()
    assert proxy[20, 5] > proxy[20, -6]  # Larger inverse prediction -> nearer.
    np.testing.assert_allclose(proxy, mapper(3 * raw + 7), atol=2e-5)
    np.testing.assert_allclose(mapper(np.ones((41, 61), np.float32)), 1 / 0.7, atol=1e-6)
    mapper.reset()
    mapper(raw, temporal=True)
    first_bounds = mapper.bounds
    mapper(raw + 10, temporal=True)
    np.testing.assert_allclose(mapper.bounds, np.asarray(first_bounds) + 1.5, atol=1e-5)
    mapper.reset()
    assert mapper.bounds is None
    # A resolution change starts a new normalization history.
    mapper(raw, temporal=True)
    mapper(np.full((20, 30), 9, dtype=np.float32), temporal=True)
    np.testing.assert_allclose(mapper.bounds, [9, 9])
    print(f"Synthetic checks passed on {device}: planes, scale, borders, unit normals, invalid Z, and proxy mapping.")


run_synthetic_checks()

Synthetic checks passed on cpu: planes, scale, borders, unit normals, invalid Z, and proxy mapping.


## Level 02 — dynamic relighting

The renderer keeps the perspective points and normals on the geometry device.
For surface point P and light Q, L = normalize(Q − P), V = normalize(−P),
and H = normalize(L + V). We negate the geometry normals to face the camera.
Diffuse is max(N·L, 0); specular is max(N·H, 0)^48, gated to lit, visible
surfaces. Falloff is power / (1 + ||Q − P||²), softened near the point light.
Shading happens in linear RGB, followed by conversion to sRGB for display.

The camera image is an approximate albedo: its existing lighting and shadows
remain. This is an added illumination effect, not recovery of intrinsic
materials. Bright values clip at white; reduce power if detail washes out.
Depth is approximate and may flicker. Cast shadows belong to level 4.


In [5]:
# Self-contained Level 2 renderer; uses the perspective points from Level 1.
def default_light():
    return dict(x=-0.6, y=-0.4, z=0.0, power=4.0, specular=0.3)


@torch.inference_mode()
def relight_rgb(frame_rgb, normals, points, valid, light=None,
                ambient=0.18, shininess=48.0, return_tensor=False):
    """RGB uint8 + perspective XYZ geometry -> relit RGB uint8.

    X right, Y down, Z away; the camera is at (0, 0, 0).
    Negate the geometry cell's normals to face the camera.
    All light coordinates use the Z proxy's arbitrary units.
    """
    light = default_light() if light is None else light
    values = [float(light[k]) for k in ("x", "y", "z", "power", "specular")]
    if (not all(math.isfinite(v) for v in values + [ambient, shininess])
            or min(values[3:]) < 0 or ambient < 0 or shininess < 1):
        raise ValueError("Light values must be finite; gains >= 0; shininess >= 1.")
    frame_rgb = np.asarray(frame_rgb)
    if (frame_rgb.ndim != 3 or frame_rgb.shape[-1] != 3
            or frame_rgb.dtype != np.uint8 or tuple(points.shape) != frame_rgb.shape
            or normals.shape != points.shape or tuple(valid.shape) != frame_rgb.shape[:2]):
        raise ValueError("Expected uint8 HxWx3 RGB and matching points, normals, validity.")
    device = points.device
    # Invalid geometry receives ambient only; sanitize before vector arithmetic.
    mask = valid.to(device=device, dtype=torch.bool)
    mask = mask & torch.isfinite(points).all(-1) & torch.isfinite(normals).all(-1)
    p = torch.nan_to_num(points.float())
    n = -F.normalize(torch.nan_to_num(normals.float()), dim=-1)
    lamp = torch.tensor(values[:3], device=device, dtype=torch.float32)
    to_light = lamp - p
    distance_sq = (to_light * to_light).sum(-1)
    l = F.normalize(to_light, dim=-1)
    v = F.normalize(-p, dim=-1)
    half_vector = F.normalize(l + v, dim=-1)
    ndotl = (n * l).sum(-1).clamp_min(0)
    facing = (n * v).sum(-1) > 0
    diffuse = torch.where(mask & facing, ndotl, 0)
    spec = (n * half_vector).sum(-1).clamp(0, 1).pow(shininess)
    spec = torch.where(mask & facing & (ndotl > 0), spec, 0)
    # Softened inverse-square falloff: finite even if the light touches a point.
    attenuation = values[3] / (1.0 + distance_sq)
    srgb = torch.as_tensor(frame_rgb, device=device).float() / 255
    albedo = torch.where(srgb <= 0.04045, srgb / 12.92,
                         ((srgb + 0.055) / 1.055).pow(2.4))
    linear = albedo * (ambient + attenuation * diffuse)[..., None]
    linear = (linear + (attenuation * values[4] * spec)[..., None]).clamp(0, 1)
    srgb = torch.where(linear <= 0.0031308, 12.92 * linear,
                       1.055 * linear.pow(1 / 2.4) - 0.055)
    result = (srgb * 255).round().clamp(0, 255).to(torch.uint8)
    return result if return_tensor else result.cpu().numpy()


In [6]:
def run_lighting_checks(device="cpu"):
    estimator = ScharrNormalEstimator(41, 61, fx=60, fy=60, device=device)
    z = torch.full((41, 61), 2.0, device=estimator.device)
    normals, points, valid = estimator.compute_normals_and_coords(z)
    rgb = np.full((41, 61, 3), 100, dtype=np.uint8)

    def render(x=0., y=0., z=0., power=4., specular=0., **kwargs):
        return relight_rgb(rgb, normals, points, valid,
                           dict(x=x, y=y, z=z, power=power, specular=specular), **kwargs)

    near, far = render(z=0.5), render(z=-2.)
    assert near[20, 30, 0] > far[20, 30, 0]
    np.testing.assert_array_equal(render(z=3., ambient=0, specular=0.5), 0)
    left, right = render(x=-0.7, specular=0.3), render(x=0.7, specular=0.3)
    np.testing.assert_allclose(left, right[:, ::-1], atol=1, rtol=0)
    assert left[20, 10, 0] > left[20, 50, 0]
    diffuse, glossy = render(), render(specular=0.5)
    delta = glossy.astype(float) - diffuse
    assert delta[20, 30, 0] > delta[20, 0, 0]
    # Coincident surface/light gives ambient only at that point, without NaN.
    assert render(z=2., ambient=0, specular=0.5)[20, 30, 0] == 0

    rgb[:] = [37, 128, 210]
    np.testing.assert_array_equal(render(power=0, ambient=1), rgb)
    np.testing.assert_array_equal(render(power=0, ambient=0), 0)
    invalid = valid.clone()
    invalid[20, 30] = False
    bad_points = points.clone()
    bad_points[20, 30] = float("nan")
    result = relight_rgb(rgb, normals, bad_points, invalid, ambient=1)
    np.testing.assert_array_equal(result[20, 30], rgb[20, 30])

    # An analytic tilted plane must favor the lamp on its outward-facing side.
    expected = F.normalize(torch.tensor([-0.6, 0., 1.], device=estimator.device), dim=0)
    tilted_z = 2.0 * expected[2] / (estimator.ray_grid * expected).sum(-1)
    normals, points, valid = estimator.compute_normals_and_coords(tilted_z)
    assert render(x=1.)[20, 30, 0] > render(x=-1.)[20, 30, 0]
    for bad in (dict(x=float("nan")), dict(power=-1)):
        settings = default_light()
        settings.update(bad)
        try:
            relight_rgb(rgb, normals, points, valid, settings)
        except ValueError:
            pass
        else:
            raise AssertionError("Invalid light settings were accepted.")
    print(f"Lighting checks passed on {device}: orientation, falloff, specular, color, masks, and tilted plane.")


run_lighting_checks()


Lighting checks passed on cpu: orientation, falloff, specular, color, masks, and tilted plane.


In [7]:
if INPUT_SIZE < 140 or INPUT_SIZE % 14:
    raise ValueError("INPUT_SIZE must be a multiple of 14 and at least 140.")
print("Loading Depth Anything V2 Small...")
processor = AutoImageProcessor.from_pretrained(MODEL_ID, use_fast=False)
processor.size = {"height": INPUT_SIZE, "width": INPUT_SIZE}
model = AutoModelForDepthEstimation.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE
).to(DEVICE).eval()
_image_mean = torch.tensor(processor.image_mean, device=DEVICE, dtype=torch.float32)[None, :, None, None]
_image_std = torch.tensor(processor.image_std, device=DEVICE, dtype=torch.float32)[None, :, None, None]


def prepare_depth_inputs(frame_rgb):
    """Match the configured aspect-preserving resize policy.

    OpenCV's cubic interpolation avoids PIL/processor bookkeeping. Its pixels
    can differ slightly from the reference PIL bicubic path; set
    FAST_PREPROCESS=False for that reference. Normalization uses model config.
    Keep resize on CPU, then upload uint8 instead of CPU float32 pixels.
    """
    if not globals().get("FAST_PREPROCESS", True):
        inputs = processor(images=frame_rgb, return_tensors="pt")
        return {key: value.to(DEVICE, DTYPE) for key, value in inputs.items()}
    h, w = frame_rgb.shape[:2]
    target_h, target_w = processor.size["height"], processor.size["width"]
    scale_h, scale_w = target_h / h, target_w / w
    if processor.keep_aspect_ratio:
        # Same choice as the HF DepthAnything image processor: resize the axis
        # whose scale is closest to 1, then round both axes to the patch grid.
        scale_h = scale_w = scale_w if abs(1-scale_w) < abs(1-scale_h) else scale_h
    multiple = processor.ensure_multiple_of
    out_h = max(multiple, int(round(h * scale_h / multiple)) * multiple)
    out_w = max(multiple, int(round(w * scale_w / multiple)) * multiple)
    resized = cv2.resize(frame_rgb, (out_w, out_h), interpolation=cv2.INTER_CUBIC)
    pixels = torch.as_tensor(resized, device=DEVICE).permute(2, 0, 1)[None].float()
    pixels = (pixels * processor.rescale_factor - _image_mean) / _image_std
    return {"pixel_values": pixels.to(DTYPE).contiguous()}


@torch.inference_mode()
def infer_relative_inverse(frame_rgb, return_tensor=False, timings=None):
    """Float inverse depth; live callers retain it on the model device."""
    if timings is not None:
        timings.begin("preprocess")
    inputs = prepare_depth_inputs(frame_rgb)
    if timings is not None:
        timings.end("preprocess")
        timings.begin("depth")
    prediction = model(**inputs).predicted_depth
    prediction = F.interpolate(
        prediction[:, None].float(), size=frame_rgb.shape[:2],
        mode="bilinear", align_corners=False,
    )[0, 0]
    if timings is not None:
        timings.end("depth")
    return prediction if return_tensor else prediction.cpu().numpy()


Loading Depth Anything V2 Small...


[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 287/287 [00:00<00:00, 3132.64it/s]

In [8]:
# Optional photo experiment. Set a local path when running outside Colab.
PHOTO_PATH = None
photo = None
if PHOTO_PATH:
    photo = Image.open(PHOTO_PATH).convert("RGB")
else:
    try:
        from google.colab import files
    except ImportError:
        print("Set PHOTO_PATH to test a local photo, or skip this cell.")
    else:
        uploaded = files.upload()
        if uploaded:
            photo = Image.open(io.BytesIO(next(iter(uploaded.values())))).convert("RGB")

if photo is not None:
    frame_rgb = np.asarray(photo)
    z_proxy = RelativeGeometryDepth()(infer_relative_inverse(frame_rgb))
    h, w = z_proxy.shape
    estimator = ScharrNormalEstimator(h, w, hfov_deg=ASSUMED_HFOV_DEG, device=DEVICE)
    normals, points, valid = estimator.compute_normals_and_coords(z_proxy)
    PHOTO_LIGHT = default_light()  # Edit x/y/z, power, specular and rerun this cell.
    relit = relight_rgb(frame_rgb, normals, points, valid, PHOTO_LIGHT)
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    axes[0].imshow(frame_rgb)
    axes[0].set_title("Original RGB")
    depth_plot = axes[1].imshow(z_proxy, cmap="magma", vmin=1 / 1.2, vmax=1 / 0.2)
    axes[1].set_title("Approximate Z — arbitrary units")
    fig.colorbar(depth_plot, ax=axes[1], fraction=0.046)
    axes[2].imshow(normals_to_rgb(normals, valid))
    axes[2].set_title("Approximate surface normals (+Z orientation)")
    axes[3].imshow(relit)
    axes[3].set_title("Level 2: virtual point light")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    r, c = h // 2, w // 2
    print(f"Center pixel ({c}, {r}): Z proxy = {z_proxy[r, c]:.3f} arbitrary units")
    print("Unit XYZ normal:", normals[r, c].cpu().numpy())
    print("These values are not a physical distance measurement.")

Set PHOTO_PATH to test a local photo, or skip this cell.


## Local live camera — Levels 01–03

The default `LIVE_BACKEND="local"` uses your webcam and an OpenCV display window.
Run this notebook with a local Python kernel (Jupyter or VS Code), then run the
live cell. The window provides X/Y/Z, power, and specular sliders. Press
**Q/Esc** or close the window to stop; **R** resets the light. If camera 0 is
unavailable, change `CAMERA_INDEX`. Install `requirements-notebook.txt` (GUI `opencv-contrib-python`) and restart the kernel.
If migrating from Level 2, run the optional install cell to remove other cv2 providers.

There is no browser JPEG upload or output encoding in local mode. A capture
thread holds only the latest camera frame, replacing unread frames so slow
inference does not build a stale queue. Every processed frame gets its own
depth, normals, and lighting; the same camera frame ID is never processed twice.
Capture age is measured from completion of the camera read, not sensor exposure.

**Local hand control is enabled by default.** Run the hand-model setup cell once
to download Google's versioned Hand Landmarker asset. Subsequent runs use the
cached file. Missing packages/model or tracking errors leave sliders available.

- Show one open, front-facing palm for three valid processed frames to acquire.
- Move right/down in the **unmirrored output** to increase light X/Y.
- Move closer to decrease Z; move farther away to increase it. First acquisition
  calibrates relative Z. Press **C** to recalibrate at a comfortable distance.
- Press **G** for hand/manual mode (or retry after an error), **R** to reset
  light and calibration, **Q/Esc** to stop. Focus the OpenCV window for keys.
- **Open your hand to brighten the light; close it into a fist to dim it.**
  Partial opening gives intermediate intensity, smoothly filtered over 180 ms.
  The power range is 0–12; zero removes the virtual light, leaving ambient light.
  Keep your palm facing the camera: rotation/foreshortening can affect openness.
- XYZ and Power sliders display the tracked values in gesture mode. Press **G**
  to edit them manually. Specular remains adjustable in either mode.
- Lost hands hold position and intensity. Clipped/invalid finger landmarks hold
  intensity while a valid palm can still control position. The window shows
  openness and power. No extra model or dependency is needed for intensity.

Local openness averages the four fingertip-to-wrist / PIP-to-wrist distance
ratios (thumb excluded), maps ratios 0.9–1.6 to 0–1, and applies smoothstep.
Distances use aspect-corrected image coordinates and are scale independent.
This is a projected openness heuristic, not a calibrated finger-angle estimate.
The optional Colab path below retains its manual Power slider.
- Missing, clipped, or tiny palms hold the last light. Reacquisition needs three
  valid frames. Resolution changes reset calibration and the VIDEO tracker.
  Use one hand; identity is not locked when multiple hands enter the frame.

The local mapper uses the same formulas and 120 ms smoothing described below
for Colab. Palm rotation can mimic Z motion; this is a relative depth gesture,
not metric hand position. Green landmark dots are drawn only on the final
output; set SHOW_HAND_LANDMARKS=False for clean relighting.

Local CPU MediaPipe VIDEO detection runs once on each processed camera frame,
before depth inference. This keeps controls, landmarks and geometry aligned.
The capture thread continues reading during processing; there is no inference
queue. Hand inference currently adds serial work, reported as the **hand** stage
and included in completed-loop FPS. Live responsiveness and FPS must be measured
on the demo machine; the browser's independent tracker rate does not apply locally.

API reference: [MediaPipe Hand Landmarker for Python](https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker/python).


Choose `PERFORMANCE_PRESET` in the setup cell. **fast** is the default.
It uses a **196** model target and **320** capture width; switch to
**balanced** for 224/480 or **detail** for 294/640.

Available resolutions:

| Setting | Fast (default) | Balanced | Detail |
|---|---|---|---|
| `INPUT_SIZE` (model target) | 196 | 224 | 294 |
| `CAPTURE_WIDTH` | 320 | 480 | 640 |

Re-run setup + model after changing model input. Restart the live cell after
changing camera/display settings. Aspect ratio is preserved. Lower resolution
reduces fine geometry and edge detail. `SHOW_DIAGNOSTICS=True` restores the
synchronized four-panel display. The local GPU determines inference speed. `GEOMETRY_BACKEND="auto"` benchmarks
CPU and GPU normalization/smoothing plus normals once at startup, then selects
the faster path for this machine and resolution. This adds a brief calibration
to warmup. You can force `"cpu"` or `"gpu"` for comparison. Normals and lighting
still run on the selected torch device. CPU smoothing requires depth transfers;
the GPU path avoids them. The startup comparison keeps this tradeoff measured.

## Optional Colab Level 03 — spatial gesture control

Run the cells above, skip the optional photo if desired, then run the live cell
and allow camera access. Keep this output visible and the browser tab active.
Enabling hand control downloads a pinned MediaPipe Tasks Vision JS/WASM runtime
(version 0.10.22) and the Hand Landmarker model in the browser. Python needs no
additional package. Manual controls work while loading and if loading fails.

1. **Hand control** follows HAND_CONTROL in setup (enabled by default). Disable it for Level 2. Show **one open palm facing the
   camera** for three tracking updates. Green dots show detected landmarks.
2. Move the palm right/left and up/down in the **unmirrored preview** to move
   light X/Y. Preview, depth, and relighting share the same orientation.
3. Move the palm **closer to the camera** to decrease light Z; move it away to
   increase Z. The first steady palm defines the current Z reference. Use
   **Calibrate hand Z** at a comfortable distance to recalibrate (also after
   changing hands). Keep the palm facing the camera: rotation changes its
   apparent size and can mimic depth motion.
4. Missing, clipped, or very small palms **hold the last light position**.
   Reacquisition requires three valid updates and resumes with smoothing.
   Use one hand; identity is not locked when multiple hands enter the frame.
5. Power and Specular remain adjustable. Uncheck **Hand control** to enable
   manual XYZ sliders. **Reset light** restores defaults and resets calibration.
   **Stop camera** or interrupt the cell to release the camera and tracker.

Mapping: palm center = mean of landmarks 0/5/9/13/17; X = 6(u−0.5),
Y = 4(v−0.5). Palm span is the 5-to-17 knuckle distance with both axes
normalized by image width. Z = reference_Z − 1.5 ln(span/reference_span),
clamped to [−2, 0.7]; X/Y are clamped to [−3, 3]/[−2, 2]. A time-based
EMA with 120 ms time constant smooths the light; a capped time step avoids
a jump after a dropout. This is a **relative gesture depth proxy**, not meters
or camera-relative depth from MediaPipe's wrist-relative landmark Z.

The browser tracks at **up to 20 updates/s**, independently of Python depth
inference. Each synchronous WASM detection can block the browser UI; this is
a rate cap, not a measured speed. There is no queue: each Python capture takes
the latest raw tracked canvas plus a copy of its light/status. Green overlay
dots stay out of model input. The default output is relighting only. Set `SHOW_DIAGNOSTICS=True` for
the synchronized 2×2 RGB/Z/normals/relit display. Capture age reports stale browser frames, while the existing
completed-cycle rate includes Colab transfers and depth/render work. Neither
is sensor-to-screen latency. Background tabs may throttle tracking.

Live Colab camera behavior and throughput must be measured on the demo machine;
this notebook does not yet implement Level 4 cast shadows.

API reference: [MediaPipe Hand Landmarker for Web](https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker/web_js).


### Hosted Colab performance controls

The setup preset also applies to Colab. Colab uses **JPEG quality 75**,
relighting-only output and status once per second. Set HAND_CONTROL=False to disable tracking.
Re-run setup and model cells when changing model settings; restart the live
cell after changing display/camera settings. Set `INPUT_SIZE=294` and
`CAPTURE_WIDTH=640` for more detail. Actual model dimensions preserve aspect
ratio and round to multiples of 14; the target is not necessarily a square.

When GPU smoothing is selected, the live path keeps floating depth,
normalization, bilateral smoothing, normals, and shading on the GPU.
Only the final uint8 image is downloaded. Auto mode can select CPU smoothing
when its measured end-to-end geometry time is faster.
The reference photo mapper remains available. GPU normalization estimates
percentiles from at most 4096 pixels; expect small differences from the full
reference percentiles. Fast preprocessing uses OpenCV cubic resize; set
`FAST_PREPROCESS=False` to compare the original Hugging Face preprocessing.

Each cycle still computes fresh depth and relighting from its captured frame.
No skipped depth updates or reused geometry inflate the reported rate.
The FPS counter excludes three warmup cycles and includes capture, inference,
encoding, and output dispatch, but not browser paint or delivery confirmation.
CUDA event stage timings avoid per-stage synchronization. `download_wait`
includes unfinished GPU work; stage timings overlap and must not be summed.
`capture` includes the Colab round trip and input JPEG decode; `display` is
output JPEG encode and dispatch. Hosted Colab transport can still limit FPS.
Actual live throughput must be measured in Colab; local tests do not prove it.


In [9]:
# Run once before the live cell for local Level 3. No camera is opened here.
# The versioned Google model is cached in the notebook working directory.
def prepare_hand_model(path):
    from pathlib import Path
    from urllib.request import urlopen
    import os
    import tempfile
    target = Path(path)
    if target.is_file():
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    url = ('https://storage.googleapis.com/mediapipe-models/'
           'hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task')
    temporary = None
    try:
        with urlopen(url, timeout=30) as response, tempfile.NamedTemporaryFile(
                dir=target.parent, suffix='.part', delete=False) as stream:
            temporary = Path(stream.name)
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                stream.write(chunk)
        if temporary.stat().st_size < 1024:
            raise RuntimeError('Hand model download was incomplete')
        os.replace(temporary, target)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()
    return target

if LIVE_BACKEND == 'local' and HAND_CONTROL:
    try:
        print('Local hand model:', prepare_hand_model(HAND_MODEL_PATH))
    except Exception as error:
        print(f'Hand model unavailable: {error}. Live sliders remain available; rerun to retry.')


Local hand model: models\hand_landmarker.task


In [10]:
def start_colab_webcam():
    from google.colab.output import eval_js
    display(Javascript(r"""
// Browser-local tracking keeps hand inference independent of Colab depth inference.
if (window.l1StopCamera) window.l1StopCamera();
window.l1Camera = {};

window.L3GestureControl = class {
    constructor() { this.reset(); }
    reset() {
        this.reference = null;
        this.lastTime = null;
        this.streak = 0;
    }
    update(landmarks, width, height, now, light) {
        const palm = landmarks && [0, 5, 9, 13, 17].map(i => landmarks[i]);
        if (!palm || palm.some(p => !p || !Number.isFinite(p.x) || !Number.isFinite(p.y)
                || p.x < 0 || p.x > 1 || p.y < 0 || p.y > 1)) {
            this.lastTime = null;
            this.streak = 0;
            return 'No hand — holding light';
        }
        // Normalize both axes by image width, so aspect ratio cannot distort size.
        const span = Math.hypot(palm[1].x - palm[4].x,
                               (palm[1].y - palm[4].y) * height / width);
        if (!Number.isFinite(span) || span < 0.025) {
            this.lastTime = null;
            this.streak = 0;
            return 'Show a larger, front-facing palm — holding light';
        }
        if (this.lastTime !== null && now - this.lastTime > 500) this.streak = 0;
        const dt = this.lastTime === null ? 50 : Math.max(0, Math.min(now - this.lastTime, 100));
        this.lastTime = now;
        if (++this.streak < 3) return 'Acquiring hand — holding light';
        if (!this.reference) this.reference = {span, z: light.z};
        const u = palm.reduce((sum, p) => sum + p.x, 0) / palm.length;
        const v = palm.reduce((sum, p) => sum + p.y, 0) / palm.length;
        const clamp = (x, lo, hi) => Math.max(lo, Math.min(hi, x));
        const target = {
            x: clamp((u - 0.5) * 6, -3, 3),
            y: clamp((v - 0.5) * 4, -2, 2),
            // A larger palm moves the light toward the camera (negative Z).
            z: clamp(this.reference.z - 1.5 * Math.log(span / this.reference.span), -2, 0.7),
        };
        const alpha = 1 - Math.exp(-dt / 120); // Time-based EMA, tau = 120 ms.
        for (const key of ['x', 'y', 'z']) light[key] += alpha * (target[key] - light[key]);
        return 'Tracking hand';
    }
};

window.l3LoadHandTracker = async function() {
    // Pin the JS and WASM to the same release. No Python MediaPipe dependency.
    const root = 'https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@0.10.22';
    const {FilesetResolver, HandLandmarker} = await import(root + '/vision_bundle.mjs');
    const files = await FilesetResolver.forVisionTasks(root + '/wasm');
    return HandLandmarker.createFromOptions(files, {
        baseOptions: {
            modelAssetPath: 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
            delegate: 'CPU',
        },
        runningMode: 'VIDEO', numHands: 1,
        minHandDetectionConfidence: 0.6,
        minHandPresenceConfidence: 0.6,
        minTrackingConfidence: 0.6,
    });
};

window.l1StopCamera = function() {
    const c = window.l1Camera;
    c.stopped = true;
    if (c.raf !== undefined) cancelAnimationFrame(c.raf);
    if (c.stream) c.stream.getTracks().forEach(track => track.stop());
    try { if (c.tracker) c.tracker.close(); }
    finally {
        if (c.wrapper) c.wrapper.remove();
        window.l1Camera = {};
    }
};

window.l1StartCamera = async function(options = {}) {
    const captureWidth = Math.max(160, Math.min(1280, options.width || 480));
    const jpegQuality = Math.max(0.01, Math.min(1, options.quality || 0.75));
    window.l1StopCamera();
    const c = window.l1Camera;
    const active = () => window.l1Camera === c && !c.stopped;
    try {
        c.wrapper = document.createElement('div');
        c.video = document.createElement('video');
        c.video.style.display = 'none';
        c.video.muted = true;
        c.video.playsInline = true;
        c.preview = document.createElement('canvas');
        c.preview.style.maxWidth = '640px';
        c.preview.style.width = '100%';
        c.status = document.createElement('p');
        c.status.setAttribute('role', 'status');
        c.light = {x: -0.6, y: -0.4, z: 0.0, power: 4.0, specular: 0.3};
        c.gesture = new window.L3GestureControl();
        c.mode = options.hand ? 'gesture' : 'manual';
        c.tracking = options.hand ? 'Loading hand model — manual sliders available' : 'Manual control';
        const controls = document.createElement('div');
        controls.style.maxWidth = '640px';
        const modeLabel = document.createElement('label');
        const mode = document.createElement('input');
        mode.type = 'checkbox'; mode.checked = !!options.hand;
        mode.setAttribute('aria-label', 'Hand control');
        modeLabel.append(mode, ' Hand control (uncheck for manual XYZ)');
        const inputs = {};
        const syncControls = () => {
            for (const key of ['x', 'y', 'z', 'power', 'specular']) {
                const {input, caption, label} = inputs[key];
                input.value = c.light[key];
                input.disabled = ['x', 'y', 'z'].includes(key) && c.mode === 'gesture' && !!c.tracker;
                caption.textContent = label + ': ' + c.light[key].toFixed(2) + ' ';
            }
            c.status.textContent = c.tracking + ' | Light XYZ: ' +
                ['x', 'y', 'z'].map(k => c.light[k].toFixed(2)).join(', ') + ' (arbitrary units)';
        };
        const addSlider = (key, label, min, max, step) => {
            const row = document.createElement('label');
            row.style.display = 'block';
            const caption = document.createElement('span');
            const input = document.createElement('input');
            input.type = 'range'; input.min = min; input.max = max; input.step = step;
            input.style.width = '60%';
            input.setAttribute('aria-label', label);
            input.oninput = () => { c.light[key] = Number(input.value); syncControls(); };
            inputs[key] = {input, caption, label, initial: c.light[key]};
            row.append(caption, input);
            controls.append(row);
        };
        addSlider('x', 'Light X (right)', -3, 3, 0.05);
        addSlider('y', 'Light Y (down)', -2, 2, 0.05);
        addSlider('z', 'Light Z (toward scene)', -2, 0.7, 0.05);
        addSlider('power', 'Power', 0, 12, 0.1);
        addSlider('specular', 'Specular', 0, 1, 0.05);
        mode.onchange = () => {
            c.mode = mode.checked ? 'gesture' : 'manual';
            c.gesture.reset();
            c.tracking = c.mode === 'manual' ? 'Manual control' :
                (c.tracker ? 'Show one palm to calibrate Z' : 'Hand model unavailable or loading — use manual sliders');
            if (c.mode === 'gesture') ensureTracker();
            syncControls();
        };
        const calibrate = document.createElement('button');
        calibrate.textContent = 'Calibrate hand Z';
        calibrate.onclick = () => {
            c.gesture.reset();
            c.tracking = 'Calibration reset — show one palm at a comfortable distance';
            syncControls();
        };
        const reset = document.createElement('button');
        reset.textContent = 'Reset light';
        reset.onclick = () => {
            for (const [key, item] of Object.entries(inputs)) c.light[key] = item.initial;
            c.gesture.reset();
            c.tracking = c.mode === 'gesture' ? 'Light reset — show one palm to calibrate Z' : 'Manual control';
            syncControls();
        };
        const stop = document.createElement('button');
        stop.textContent = 'Stop camera'; stop.onclick = window.l1StopCamera;
        c.wrapper.append(c.video, c.preview, c.status, modeLabel, controls, calibrate, reset, stop);
        document.body.append(c.wrapper);
        syncControls();
        c.stream = await navigator.mediaDevices.getUserMedia({
            video: {width: {ideal: 640}, height: {ideal: 480}}, audio: false,
        });
        if (!active()) { c.stream.getTracks().forEach(t => t.stop()); return false; }
        c.video.srcObject = c.stream;
        await c.video.play();
        if (!active()) return false;
        if (!c.video.videoWidth || !c.video.videoHeight) throw new Error('Camera returned invalid dimensions');
        c.canvas = document.createElement('canvas');
        c.lastVideoTime = -1;
        c.lastTick = -Infinity;
        c.frameId = 0;
        c.updateFrame = now => {
            if (!active() || c.video.readyState < 2 || c.video.currentTime === c.lastVideoTime) return;
            const scale = Math.min(1, captureWidth / c.video.videoWidth);
            const width = Math.max(3, Math.round(c.video.videoWidth * scale));
            const height = Math.max(3, Math.round(c.video.videoHeight * scale));
            if (c.canvas.width !== width || c.canvas.height !== height) {
                c.canvas.width = c.preview.width = width;
                c.canvas.height = c.preview.height = height;
                c.gesture.reset();
            }
            c.canvas.getContext('2d').drawImage(c.video, 0, 0, width, height);
            c.lastVideoTime = c.video.currentTime;
            c.capturedAt = now;
            c.frameId++;
            let landmarks;
            if (c.tracker && c.mode === 'gesture') {
                try {
                    landmarks = c.tracker.detectForVideo(c.canvas, now).landmarks[0];
                    c.tracking = c.gesture.update(landmarks, width, height, now, c.light);
                } catch (error) {
                    const failed = c.tracker;
                    c.tracker = null;
                    try { failed.close(); } catch (_) {}
                    c.mode = 'manual'; mode.checked = false;
                    c.tracking = 'Hand tracking failed — manual control: ' + error.message;
                }
            }
            // Preview landmarks never enter the RGB image sent to the depth model.
            const ctx = c.preview.getContext('2d');
            ctx.drawImage(c.canvas, 0, 0);
            if (landmarks) {
                ctx.fillStyle = '#00ffb3';
                for (const point of landmarks) {
                    ctx.beginPath(); ctx.arc(point.x * width, point.y * height, 3, 0, 2 * Math.PI); ctx.fill();
                }
            }
            syncControls();
        };
        c.updateFrame(performance.now());
        const tick = now => {
            if (!active()) return;
            // At most 20 tracking updates/s; synchronous WASM can still block the UI.
            if (now - c.lastTick >= 50) { c.lastTick = now; c.updateFrame(now); }
            c.raf = requestAnimationFrame(tick);
        };
        c.raf = requestAnimationFrame(tick);
        // Camera/manual controls work while the assets load, or if loading fails.
        function ensureTracker() {
            if (c.tracker || c.loading) return;
            c.loading = window.l3LoadHandTracker().then(tracker => {
            if (!active()) { tracker.close(); return; }
            c.tracker = tracker;
            c.tracking = c.mode === 'gesture' ? 'Show one palm to calibrate Z' : 'Manual control';
            syncControls();
        }).catch(error => {
            if (!active()) return;
            c.mode = 'manual'; mode.checked = false;
            c.tracking = 'Hand model could not load — manual control: ' + error.message;
            syncControls();
            }).finally(() => { c.loading = null; });
        };
        c.jpegQuality = jpegQuality;
        if (c.mode === 'gesture') ensureTracker();
        return true;
    } catch (error) {
        if (active()) window.l1StopCamera();
        throw error;
    }
};

window.l1CaptureFrame = function() {
    const c = window.l1Camera;
    if (!c.stream || !c.canvas || !c.stream.getVideoTracks().some(t => t.readyState === 'live')) return null;
    // Manual Level 2 captures can use a fresh video frame, beyond the 20 Hz tracker cap.
    if (c.mode === 'manual') c.updateFrame(performance.now());
    // Snapshot the last tracked canvas and its control state atomically; no frame queue.
    return {
        image: c.canvas.toDataURL('image/jpeg', c.jpegQuality), light: {...c.light},
        hand: {mode: c.mode, status: c.tracking, frame_id: c.frameId,
               age_ms: Math.max(0, performance.now() - c.capturedAt)},
    };
};

    """))
    import json
    options = dict(width=CAPTURE_WIDTH, quality=JPEG_QUALITY / 100, hand=HAND_CONTROL)
    return eval_js("l1StartCamera(" + json.dumps(options) + ")")


def get_colab_frame():
    from google.colab.output import eval_js
    data = eval_js("l1CaptureFrame()")
    if not data:
        return None
    binary = b64decode(data["image"].split(",", 1)[1])
    bgr = cv2.imdecode(np.frombuffer(binary, dtype=np.uint8), cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError("Could not decode the webcam frame.")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), data["light"], data["hand"]


def stop_colab_webcam():
    from google.colab.output import eval_js
    eval_js("window.l1StopCamera && window.l1StopCamera()")


def live_panel(frame_rgb, z_proxy, normal_rgb, relit_rgb):
    scaled = np.clip((z_proxy - 1 / 1.2) / (1 / 0.2 - 1 / 1.2), 0, 1)
    depth_rgb = cv2.cvtColor(
        cv2.applyColorMap((scaled * 255).astype(np.uint8), cv2.COLORMAP_MAGMA),
        cv2.COLOR_BGR2RGB,
    )
    panels = []
    for picture, title in zip(
        (frame_rgb, depth_rgb, normal_rgb, relit_rgb),
        ("Processed RGB", "Z proxy (arbitrary units)", "Approximate normals", "Level 3: hand-controlled light"),
    ):
        panel = cv2.copyMakeBorder(picture, 32, 0, 0, 0, cv2.BORDER_CONSTANT)
        cv2.putText(panel, title, (8, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        panels.append(panel)
    top = np.concatenate(panels[:2], axis=1)
    bottom = np.concatenate(panels[2:], axis=1)
    return Image.fromarray(np.concatenate((top, bottom), axis=0))


def hand_openness(landmarks, width, height):
    """Projected finger extension, normalized by wrist-to-PIP length.

    Average four fingers; thumb position does not change intensity. Both axes
    use image-width units so distance and camera aspect ratio cancel out.
    Invalid/clipped landmarks hold intensity rather than switching it off.
    """
    if landmarks is None or len(landmarks) < 21:
        return None
    indices = (0, 6, 8, 10, 12, 14, 16, 18, 20)
    xy = np.asarray([[landmarks[i].x, landmarks[i].y] for i in indices], dtype=np.float64)
    if not np.isfinite(xy).all() or (xy < 0).any() or (xy > 1).any():
        return None
    xy[:, 1] *= height / width
    pip_distance = np.linalg.norm(xy[1::2] - xy[0], axis=1)
    if (pip_distance < 1e-6).any():
        return None
    tip_distance = np.linalg.norm(xy[2::2] - xy[0], axis=1)
    extension = np.clip((tip_distance / pip_distance - 0.9) / 0.7, 0, 1).mean()
    return float(extension * extension * (3 - 2 * extension))


def count_extended_fingers(landmarks, width, height):
    """Count extended fingertips with finger ratios and a thumb bend angle.

    Same normalized, aspect-corrected geometry as hand_openness: both image
    axes use width units so distance and camera aspect ratio cancel out.
    A finger counts as up when its smoothstep extension exceeds 0.5.
    Returns (count, per-finger smoothstep values) or None on bad landmarks.
    """
    if landmarks is None or len(landmarks) < 21:
        return None
    joints = (0, 3, 6, 10, 14, 18, 4, 8, 12, 16, 20)
    xy = np.asarray([[landmarks[i].x, landmarks[i].y] for i in joints], dtype=np.float64)
    if not np.isfinite(xy).all() or (xy < 0).any() or (xy > 1).any():
        return None
    xy[:, 1] *= height / width
    wrist = xy[0]
    pips, tips = xy[1:6], xy[6:11]
    smooth = []
    # The thumb bends in a different plane, so the finger ratio
    # underestimates it and makes an extended thumb look down.
    thumb_mcp = np.asarray([landmarks[2].x, landmarks[2].y], dtype=np.float64)
    thumb_ip = np.asarray([landmarks[3].x, landmarks[3].y], dtype=np.float64)
    thumb_tip = np.asarray([landmarks[4].x, landmarks[4].y], dtype=np.float64)
    thumb_mcp[1] *= height / width
    thumb_ip[1] *= height / width
    thumb_tip[1] *= height / width
    thumb_v1 = thumb_mcp - thumb_ip
    thumb_v2 = thumb_tip - thumb_ip
    thumb_denom = np.linalg.norm(thumb_v1) * np.linalg.norm(thumb_v2)
    if thumb_denom < 1e-9:
        return None
    thumb_angle = np.arccos(np.clip(np.dot(thumb_v1, thumb_v2) / thumb_denom, -1, 1))
    thumb_extension = np.clip((thumb_angle - np.deg2rad(90)) / np.deg2rad(60), 0, 1)
    smooth.append(float(thumb_extension * thumb_extension * (3 - 2 * thumb_extension)))
    for pip, tip in zip(pips[1:], tips[1:]):
        denom = max(np.linalg.norm(pip - wrist), 1e-9)
        extension = np.clip((np.linalg.norm(tip - wrist) / denom - 0.9) / 0.7, 0, 1)
        smooth.append(float(extension * extension * (3 - 2 * extension)))
    return sum(v > 0.5 for v in smooth), smooth


class PalmLightControl:
    """Map a front-facing palm to relative XYZ; never interpret landmark Z as meters."""
    def __init__(self):
        self.reset()

    def reset(self):
        self.reference = None
        self.last_time = None
        self.streak = 0
        self.shape = None

    def update(self, landmarks, width, height, now, light):
        if self.shape != (height, width):
            self.reset()
            self.shape = (height, width)
        palm = None
        if landmarks is not None and len(landmarks) >= 21:
            palm = np.asarray([[landmarks[i].x, landmarks[i].y]
                               for i in (0, 5, 9, 13, 17)], dtype=np.float64)
        if (palm is None or not np.isfinite(palm).all()
                or (palm < 0).any() or (palm > 1).any()):
            self.last_time, self.streak = None, 0
            return "No valid palm - holding light"
        span = float(np.hypot(palm[1, 0] - palm[4, 0],
                              (palm[1, 1] - palm[4, 1]) * height / width))
        if span < 0.025:
            self.last_time, self.streak = None, 0
            return "Show a larger front-facing palm - holding light"
        if self.last_time is not None and now - self.last_time > 0.5:
            self.streak = 0
        dt = 0.05 if self.last_time is None else np.clip(now - self.last_time, 0, 0.1)
        self.last_time = now
        self.streak += 1
        if self.streak < 3:
            return "Acquiring palm - holding light"
        if self.reference is None:
            self.reference = (span, light['z'])
        u, v = palm.mean(axis=0)
        target = (np.clip(6 * (u - 0.5), -3, 3),
                  np.clip(4 * (v - 0.5), -2, 2),
                  np.clip(self.reference[1] - 1.5 * np.log(span / self.reference[0]), -2, 0.7))
        alpha = 1 - np.exp(-dt / 0.12)
        for key, value in zip(('x', 'y', 'z'), target):
            light[key] = float(light[key] + alpha * (value - light[key]))
        openness = hand_openness(landmarks, width, height)
        if openness is None:
            return "Tracking palm | intensity held"
        # Closed -> 0, open -> 12; ambient scene lighting remains visible.
        power_alpha = 1 - np.exp(-dt / 0.18)
        light['power'] = float(np.clip(light['power'] + power_alpha *
                                      (12.0 * openness - light['power']), 0, 12))
        return f"Tracking palm | open {openness:.0%} | power {light['power']:.1f}"


class LocalHandTracker:
    """Synchronous VIDEO detection on the exact RGB frame used for depth."""
    def __init__(self, model_path):
        from pathlib import Path
        try:
            import mediapipe as mp
        except ImportError as error:
            raise RuntimeError('Install requirements-notebook.txt and restart the kernel') from error
        if not Path(model_path).is_file():
            raise RuntimeError('Hand model missing; run the hand-model setup cell')
        self.mp = mp
        self.last_timestamp = -1
        options = mp.tasks.vision.HandLandmarkerOptions(
            base_options=mp.tasks.BaseOptions(model_asset_path=str(model_path),
                                             delegate=mp.tasks.BaseOptions.Delegate.CPU),
            running_mode=mp.tasks.vision.RunningMode.VIDEO, num_hands=1,
            min_hand_detection_confidence=0.6, min_hand_presence_confidence=0.6,
            min_tracking_confidence=0.6,
        )
        self.tracker = mp.tasks.vision.HandLandmarker.create_from_options(options)

    def detect(self, rgb, now):
        # Millisecond rounding must not create duplicate timestamps.
        timestamp = max(self.last_timestamp + 1, int(now * 1000))
        self.last_timestamp = timestamp
        image = self.mp.Image(image_format=self.mp.ImageFormat.SRGB,
                              data=np.ascontiguousarray(rgb))
        result = self.tracker.detect_for_video(image, timestamp)
        return result.hand_landmarks[0] if result.hand_landmarks else None

    def close(self):
        if self.tracker is not None:
            self.tracker.close()
            self.tracker = None


class LocalGestureSession:
    """Own light/calibration state and fail back to manual controls on errors."""
    def __init__(self):
        self.light = default_light()
        self.gesture = PalmLightControl()
        self.tracker = None
        self.enabled = False
        self.status = 'Manual XYZ sliders'
        self.shape = None

    def enable(self):
        try:
            if self.tracker is None:
                self.tracker = LocalHandTracker(globals().get('HAND_MODEL_PATH', 'models/hand_landmarker.task'))
            self.enabled = True
            self.calibrate()
        except Exception as error:
            self.enabled = False
            self.status = f'Hand tracking unavailable: {error}. G: retry; sliders active'
            print(self.status)

    def calibrate(self):
        self.gesture.reset()
        self.status = 'Show one palm to calibrate Z' if self.enabled else 'Manual XYZ sliders'

    def toggle(self):
        if self.enabled:
            self.enabled = False
            self.calibrate()
        else:
            self.enable()

    def update(self, rgb, sliders):
        start = time.perf_counter()
        landmarks = None
        if self.enabled:
            self.light.update(specular=sliders['specular'])
            try:
                # Recreate VIDEO tracking after a resolution change, not just the Z reference.
                if self.shape is not None and self.shape != rgb.shape[:2]:
                    self.close()
                    self.enable()
                self.shape = rgb.shape[:2]
                if self.enabled:
                    landmarks = self.tracker.detect(rgb, start)
                    self.status = self.gesture.update(landmarks, rgb.shape[1], rgb.shape[0],
                                                      start, self.light)
            except Exception as error:
                self.enabled = False
                try:
                    self.close()
                except Exception as close_error:
                    print(f'Hand tracker cleanup failed: {close_error}')
                self.status = f'Tracking failed: {error}. G: retry; sliders active'
                print(self.status)
                landmarks = None
        else:
            self.light = dict(sliders)
        return dict(self.light), dict(mode='gesture' if self.enabled else 'manual',
                                      status=self.status, landmarks=landmarks,
                                      hand_ms=(time.perf_counter() - start) * 1000)

    def close(self):
        self.enabled = False
        if self.tracker is not None:
            self.tracker.close()
            self.tracker = None


def sync_local_controls(light):
    for key, label, low, high, step in local_light_specs():
        if key in ('x', 'y', 'z', 'power'):
            cv2.setTrackbarPos(label, _local_window, round((light[key] - low) / step))


def annotate_local_hand(picture, hand, height, width):
    """Draw only on the output copy, keeping overlays out of model inputs."""
    points = hand.get('landmarks')
    if globals().get('SHOW_HAND_LANDMARKS', True) and points is not None:
        ox, oy = (width, height + 64) if SHOW_DIAGNOSTICS else (0, 0)
        for p in points:
            if np.isfinite([p.x, p.y]).all() and 0 <= p.x <= 1 and 0 <= p.y <= 1:
                cv2.circle(picture, (ox + round(p.x * (width - 1)), oy + round(p.y * (height - 1))),
                           2, (0, 255, 160), -1)
    cv2.putText(picture, hand['status'][:90], (8, 36),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 160), 1, cv2.LINE_AA)
    fingers = count_extended_fingers(points, width, height)
    if fingers is not None:
        cv2.putText(picture, f"fingers up: {fingers[0]}/5", (8, 54),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 160), 1, cv2.LINE_AA)



class LatestLocalCamera:
    """One latest frame, no inference backlog; camera read stays off the UI thread."""
    def __init__(self, index=0, width=480, fps=30):
        import threading
        import sys
        if width < 3 or fps <= 0:
            raise ValueError("Camera width must be >=3 and FPS positive.")
        backend = cv2.CAP_DSHOW if sys.platform == "win32" else cv2.CAP_ANY
        self.capture = cv2.VideoCapture(index, backend)
        if not self.capture.isOpened():
            self.capture.release()
            raise RuntimeError(f"Could not open camera {index}; close other camera apps or change CAMERA_INDEX.")
        self.capture.set(cv2.CAP_PROP_FRAME_WIDTH, width)
        self.capture.set(cv2.CAP_PROP_FRAME_HEIGHT, round(width * 0.75))
        self.capture.set(cv2.CAP_PROP_FPS, fps)
        self.capture.set(cv2.CAP_PROP_BUFFERSIZE, 1)  # Best effort; some drivers ignore it.
        self.width = width
        self.condition = threading.Condition()
        self.stopped = False
        self.latest = None
        self.sequence = self.delivered = 0
        self.error = None
        self.thread = threading.Thread(target=self._read, name="vision-camera", daemon=True)
        self.thread.start()

    def _read(self):
        try:
            while not self.stopped:
                ok, bgr = self.capture.read()
                if not ok or bgr is None:
                    raise RuntimeError("Camera stopped returning frames.")
                captured_at = time.perf_counter()
                h, w = bgr.shape[:2]
                if w > self.width:
                    bgr = cv2.resize(bgr, (self.width, max(3, round(h*self.width/w))),
                                     interpolation=cv2.INTER_AREA)
                rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                with self.condition:
                    if self.stopped:
                        break
                    self.sequence += 1
                    self.latest = (rgb, captured_at, self.sequence)
                    self.condition.notify_all()
        except Exception as error:
            with self.condition:
                self.error = error
                self.condition.notify_all()
        finally:
            self.capture.release()

    def get(self, timeout=5.0):
        with self.condition:
            ready = self.condition.wait_for(
                lambda: self.stopped or self.error is not None or self.sequence > self.delivered,
                timeout=timeout,
            )
            if self.stopped:
                return None
            if self.error is not None:
                raise self.error
            if not ready:
                raise RuntimeError("Timed out waiting for a new camera frame.")
            rgb, captured_at, sequence = self.latest
            self.delivered = sequence
            return rgb, {"mode": "manual", "status": "Local XYZ sliders", "frame_id": sequence,
                         "age_ms": max(0, (time.perf_counter()-captured_at)*1000)}

    def close(self):
        with self.condition:
            self.stopped = True
            self.condition.notify_all()
        self.thread.join(timeout=1.0)
        if self.thread.is_alive():
            self.capture.release()  # Unblock drivers that support cancelling read.
            self.thread.join(timeout=1.0)
        if self.thread.is_alive():
            raise RuntimeError("Camera driver did not stop; restart the kernel before reopening the camera.")


def is_local_backend():
    backend = globals().get("LIVE_BACKEND", "local")
    if backend not in ("local", "colab"):
        raise ValueError('LIVE_BACKEND must be "local" or "colab".')
    return backend == "local"


def local_light_specs():
    return [("x", "X", -3., 3., 0.05), ("y", "Y", -2., 2., 0.05),
            ("z", "Z", -2., 0.7, 0.05), ("power", "Power", 0., 12., 0.1),
            ("specular", "Specular", 0., 1., 0.05)]


def reset_local_light():
    values = default_light()
    session = globals().get("_local_gesture")
    if session is not None:
        session.light = dict(values)
        session.calibrate()
    for key, label, low, high, step in local_light_specs():
        cv2.setTrackbarPos(label, _local_window, round((values[key]-low)/step))


def start_webcam():
    if not is_local_backend():
        return start_colab_webcam()
    global _local_camera, _local_window, _local_gesture
    if globals().get("_local_camera") is not None or globals().get("_local_gesture") is not None:
        stop_webcam()
    _local_camera = None
    _local_gesture = LocalGestureSession()
    _local_window = "AI Vision - Level 3 (G: hand/manual, C: calibrate, R: reset, Q: stop)"
    try:
        _local_camera = LatestLocalCamera(CAMERA_INDEX, CAPTURE_WIDTH, CAMERA_FPS)
        cv2.namedWindow(_local_window, cv2.WINDOW_NORMAL)
        cv2.resizeWindow(_local_window, max(640, CAPTURE_WIDTH), 540)
        for key, label, low, high, step in local_light_specs():
            cv2.createTrackbar(label, _local_window, 0, round((high-low)/step), lambda _: None)
        reset_local_light()
        if globals().get("HAND_CONTROL", False):
            _local_gesture.enable()
        return True
    except Exception:
        stop_webcam()
        raise


def local_window_open():
    try:
        return cv2.getWindowProperty(_local_window, cv2.WND_PROP_VISIBLE) >= 1
    except cv2.error:
        return False


def get_frame():
    if not is_local_backend():
        return get_colab_frame()
    if not local_window_open():
        return None
    frame = _local_camera.get()
    if frame is None:
        return None
    rgb, hand = frame
    light = {key: low + cv2.getTrackbarPos(label, _local_window)*step
             for key, label, low, high, step in local_light_specs()}
    light, tracking = _local_gesture.update(rgb, light)
    hand.update(tracking)
    hand["age_ms"] += tracking["hand_ms"]
    sync_local_controls(light)
    return rgb, light, hand


def show_local_frame(rgb):
    if not local_window_open():
        return False
    cv2.imshow(_local_window, cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    key = cv2.waitKey(1) & 0xFF
    if key in (ord("q"), 27):
        return False
    if key == ord("r"):
        reset_local_light()
    elif key == ord("g"):
        _local_gesture.toggle()
    elif key == ord("c"):
        _local_gesture.calibrate()
    return local_window_open()


def stop_webcam():
    if not is_local_backend():
        return stop_colab_webcam()
    global _local_camera, _local_gesture
    camera = globals().get("_local_camera")
    try:
        if camera is not None:
            camera.close()
    finally:
        try:
            session = globals().get("_local_gesture")
            if session is not None:
                session.close()
                _local_gesture = None
        finally:
            close_local_window(camera)


def close_local_window(camera):
    global _local_camera
    # Retain an unclosed camera reference so a rerun retries its cleanup.
    if camera is None or not camera.thread.is_alive():
        _local_camera = None
    try:
        cv2.destroyWindow(globals().get("_local_window", "AI Vision"))
        cv2.waitKey(1)
    except Exception:
        pass


class StageTimings:
    """CPU wall timings + CUDA event intervals without per-stage barriers.

    GPU event times include stream idle time between events (e.g. preprocessing).
    Display timing is encoding/dispatch, not network delivery or browser paint.
    Resolve only after the final blocking image download completes.
    """
    def __init__(self, device, enabled=True):
        self.cuda = enabled and torch.device(device).type == "cuda"
        self.enabled = enabled
        self.events, self.starts, self.ms = {}, {}, {}

    def begin(self, name):
        if not self.enabled:
            return
        self.starts[name] = time.perf_counter()
        if self.cuda:
            pair = self.events.setdefault(name, (torch.cuda.Event(enable_timing=True),
                                                  torch.cuda.Event(enable_timing=True)))
            pair[0].record()

    def end(self, name):
        if not self.enabled:
            return
        self.ms[name] = (time.perf_counter() - self.starts[name]) * 1000
        if self.cuda:
            self.events[name][1].record()

    def resolve(self):
        return {name: a.elapsed_time(b) for name, (a, b) in self.events.items()} if self.cuda else dict(self.ms)


def encode_live_image(rgb):
    # Explicit JPEG avoids IPython's implicit PNG encoding of PIL objects.
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    ok, encoded = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
    if not ok:
        raise RuntimeError("Could not encode the rendered frame.")
    return DisplayImage(data=encoded.tobytes(), format="jpeg")


def compute_live_geometry(raw, mapper, estimator):
    if isinstance(mapper, DeviceRelativeGeometryDepth):
        depth = mapper(raw, temporal=True, validate=False)
    else:
        depth = mapper(raw.cpu().numpy(), temporal=True)
    depth = torch.as_tensor(depth, device=estimator.device, dtype=torch.float32)
    normals, points, valid = estimator.compute_normals_and_coords(depth, validate=False)
    return depth, normals, points, valid


def select_geometry_mapper(raw, estimator):
    """Benchmark complete normalization+normals paths once per frame shape.

    CPU OpenCV can beat GPU tensor smoothing on smaller GPUs. Warmup and
    synchronization are confined to startup; steady-state frames have no
    calibration barriers. Return a fresh mapper so profiling cannot alter EMA.
    """
    choice = globals().get("GEOMETRY_BACKEND", "auto")
    factories = {"cpu": RelativeGeometryDepth, "gpu": DeviceRelativeGeometryDepth}
    if choice not in ("auto", "cpu", "gpu"):
        raise ValueError('GEOMETRY_BACKEND must be "auto", "cpu", or "gpu".')
    if choice != "auto":
        return factories[choice](), choice
    if raw.device.type != "cuda":
        return RelativeGeometryDepth(), "cpu"
    durations = {}
    for name, factory in factories.items():
        candidate = factory()
        for _ in range(2):
            compute_live_geometry(raw, candidate, estimator)
        torch.cuda.synchronize(raw.device)
        start = time.perf_counter()
        for _ in range(5):
            compute_live_geometry(raw, candidate, estimator)
        torch.cuda.synchronize(raw.device)
        durations[name] = (time.perf_counter()-start)*1000/5
    selected = min(durations, key=durations.get)
    print(f"Geometry auto: CPU {durations['cpu']:.1f} ms, GPU {durations['gpu']:.1f} ms; using {selected.upper()} smoothing.")
    return factories[selected](), selected


def run_live():
    status = display(Markdown("Starting camera..."), display_id=True)
    output = None if is_local_backend() else display(DisplayImage(data=b"", format="jpeg"), display_id=True)
    if is_local_backend() and not all(hasattr(cv2, name) for name in
                                      ("CAP_DSHOW", "VideoCapture", "namedWindow", "imshow", "waitKey")):
        status.update(Markdown("**Live camera skipped:** this OpenCV build has no local GUI/camera support."))
        return
    mapper, geometry_backend = None, "pending"
    estimator = None
    cycles, stage_samples = deque(maxlen=60), deque(maxlen=60)
    timer = StageTimings(DEVICE, PROFILE_STAGES)
    completed, last_status = 0, -float("inf")
    try:
        if not start_webcam():
            return
        while True:
            start = time.perf_counter()
            packet = get_frame()
            if packet is None:
                break
            capture_ms = (time.perf_counter() - start) * 1000
            frame_rgb, light, hand = packet
            hand_ms = hand.get("hand_ms", 0.0)
            capture_ms = max(0.0, capture_ms - hand_ms)
            h, w = frame_rgb.shape[:2]
            if estimator is None or (estimator.H, estimator.W) != (h, w):
                estimator = ScharrNormalEstimator(h, w, hfov_deg=ASSUMED_HFOV_DEG, device=DEVICE)
                mapper = None
            raw = infer_relative_inverse(frame_rgb, return_tensor=True, timings=timer)
            if mapper is None:
                mapper, geometry_backend = select_geometry_mapper(raw, estimator)
            timer.begin("geometry")
            z_proxy, normals, points, valid = compute_live_geometry(raw, mapper, estimator)
            timer.end("geometry")
            timer.begin("lighting")
            relit = relight_rgb(frame_rgb, normals, points, valid, light, return_tensor=True)
            timer.end("lighting")
            download_start = time.perf_counter()
            if SHOW_DIAGNOSTICS:
                # One packed download for all GPU-generated debug panels.
                normal_rgb = ((normals + 1) * 127.5).round().clamp(0, 255).to(torch.uint8)
                normal_rgb = torch.where(valid[..., None], normal_rgb, torch.zeros_like(normal_rgb))
                z_byte = ((z_proxy - 1/1.2) / (1/0.2 - 1/1.2) * 255).clamp(0, 255).to(torch.uint8)
                packed = torch.cat((relit, normal_rgb, z_byte[..., None]), -1).cpu().numpy()
                # live_panel accepts a float proxy; reconstruct only for its colormap.
                debug_z = packed[..., 6].astype(np.float32) / 255 * (1/0.2 - 1/1.2) + 1/1.2
                picture = np.asarray(live_panel(frame_rgb, debug_z, packed[..., 3:6], packed[..., :3]))
            else:
                picture = relit.cpu().numpy()
            download_ms = (time.perf_counter() - download_start) * 1000
            stage_ms = timer.resolve()  # Last .cpu() has already completed the CUDA events.
            display_start = time.perf_counter()
            if is_local_backend():
                picture = picture.copy()
                annotate_local_hand(picture, hand, h, w)
                cv2.putText(picture, f"{w}x{h} | Q: stop, R: reset",
                            (8, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 160), 1, cv2.LINE_AA)
                if not show_local_frame(picture):
                    break
            else:
                output.update(encode_live_image(picture))
            stage_ms.update(capture=capture_ms, hand=hand_ms, download_wait=download_ms,
                            display=(time.perf_counter() - display_start) * 1000)
            now = time.perf_counter()
            if now - last_status >= STATUS_INTERVAL:
                last_status = now
                if stage_samples:
                    keys = ("capture", "hand", "preprocess", "depth", "geometry", "lighting", "download_wait", "display")
                    timing_text = " | ".join(f"{key}: {np.mean([s[key] for s in stage_samples if key in s]):.1f} ms"
                                             for key in keys if any(key in s for s in stage_samples))
                else:
                    timing_text = "Collecting stage timings after warmup"
                status.update(Markdown(
                    f"**Stages ({geometry_backend.upper()} smoothing):** {timing_text}  \n"
                    "CUDA stages overlap CPU work; download_wait includes queued GPU work. Do not add these timings.  \n"
                    f"**Light XYZ:** ({light['x']:+.2f}, {light['y']:+.2f}, {light['z']:+.2f}) arbitrary units | "
                    f"power: {light['power']:.1f} | specular: {light['specular']:.2f}  \n"
                    f"**Hand control:** {hand['mode']} | {hand['status']} | capture age: {hand['age_ms']:.0f} ms  \n"
                    + ("Move one palm for XYZ. **G**: hand/manual; **C**: recalibrate Z; **R**: reset; **Q/Esc**: stop. Open/close your hand for intensity. Specular stays adjustable; G enables manual power."
                     if is_local_backend() else
                     "Use XYZ sliders for Level 2; enable **Hand control** for Level 3. **Stop camera** to finish.")
                ))
            completed += 1
            if completed > 3:
                cycles.append(time.perf_counter() - start)
                stage_samples.append(stage_ms)
    except KeyboardInterrupt:
        pass
    finally:
        try:
            stop_webcam()
        except Exception as cleanup_error:
            print(f"Camera cleanup could not be confirmed: {cleanup_error}")
        rate = f" Last measured loop rate: {len(cycles) / sum(cycles):.2f} cycles/s." if cycles else ""
        status.update(Markdown(f"**Processing stopped.**{rate}"))


run_live()


**Live camera skipped:** this OpenCV build has no local GUI/camera support.

## Optional ONNX experiment — original results preserved

Run the earlier setup, geometry, lighting and model cells first. This separate experiment
exports the **current preset's actual tensor dimensions**, verifies CUDA execution, binds
GPU inputs/outputs, checks depth/normal/relighting differences, and compares warmed-up
PyTorch FP16 and ONNX timings on identical inputs in alternating order.

The default uses synthetic images and reports **compute throughput, not live camera FPS**.
Set `ONNX_IMAGE_PATH` to a scene photo for a more useful visual comparison. The quality
image saved alongside the JSON is **RGB | PyTorch relighting | ONNX relighting**.
Numerical thresholds are experiment guardrails; inspect real faces, hands and edges too.

For a real FPS comparison, run and stop the original local live cell once (to define its
camera/gesture helpers), then set **`ONNX_LIVE_COMPARE = True`** below. It runs 150 measured
frames per backend after 10 warmups, with your current hand and diagnostic settings.
**Q/Esc cancels**; G/C/R and sliders work as before. The two live runs observe different
camera frames, so maintain the same scene and hand mode. Camera/display and hand tracking
are included in live FPS. Missing hand tracking is reported as manual, not hidden.

This cell never replaces the original inference function or its saved outputs. Each run
writes a fresh `models/onnx_experiments/<timestamp>/` directory with the export, provider
profile, quality image and measurements. Export/session startup are excluded from timing.
If CUDA or quality checks fail, the experiment stops and the original backend remains usable.
Rerun for each preset; a camera aspect-ratio change requires a matching export.

One-time dependencies in the notebook kernel (CUDA 12/cuDNN 9, including this MX550 setup):
`%pip install onnx==1.17.0 onnxruntime-gpu==1.20.2 "protobuf<5"`
Keep the installed CUDA PyTorch build. No automatic package installation occurs in this cell.

References: [CUDA compatibility and streams](https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html),
[GPU I/O binding](https://onnxruntime.ai/docs/performance/tune-performance/iobinding.html).


In [11]:
# Optional, isolated ONNX experiment. Run setup/geometry/lighting/model cells first.
# For live A/B, run the existing live cell once, stop with Q, then enable below.
# Missing dependencies: %pip install onnx==1.17.0 onnxruntime-gpu==1.20.2 "protobuf<5"
ONNX_LIVE_COMPARE = False  # True: 150 frames of PyTorch, then 150 of ONNX; Q stops.
ONNX_IMAGE_PATH = None  # Optional real scene photo; None uses labeled synthetic inputs.
ONNX_BENCHMARK_FRAMES = 30
ONNX_LIVE_FRAMES = 150


def run_onnx_experiment(live=False, image_path=None, count=30, live_frames=150):
    """Export + paired CUDA timings; preserve the baseline model/functions/settings.

    Results are compute throughput unless explicitly labeled live completed-loop FPS.
    Each invocation writes a separate models/onnx_experiments/<timestamp> directory.
    """
    import copy
    import datetime
    import json
    import time
    from pathlib import Path
    import numpy as np
    import torch  # Load PyTorch CUDA/cuDNN DLLs before importing ONNX Runtime.
    import torch.nn.functional as F
    try:
        import onnx
        import onnxruntime as ort
    except ImportError as error:
        raise RuntimeError('Install onnx==1.17.0 onnxruntime-gpu==1.20.2 "protobuf<5" in this kernel.') from error

    required = ('model', 'processor', 'prepare_depth_inputs', 'infer_relative_inverse',
                'ScharrNormalEstimator', 'RelativeGeometryDepth', 'DeviceRelativeGeometryDepth', 'relight_rgb')
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(f'Run the earlier setup, geometry, lighting and model cells first: {missing}')
    if DEVICE.type != 'cuda' or not torch.cuda.is_available():
        raise RuntimeError('This experiment requires the CUDA notebook kernel; baseline is unchanged.')
    if not str(torch.version.cuda).startswith('12.') or torch.backends.cudnn.version() < 90000:
        raise RuntimeError('This pinned ONNX Runtime build expects CUDA 12 and cuDNN 9.')
    if model.training:
        raise RuntimeError('Run the model setup cell first; the baseline must already be in eval mode.')
    if 'CUDAExecutionProvider' not in ort.get_available_providers():
        raise RuntimeError('CUDAExecutionProvider unavailable. Install onnxruntime-gpu, not the CPU package.')
    if count < 2 or live_frames < 1:
        raise ValueError('Use at least 2 benchmark frames and 1 live frame.')
    if live and (globals().get('LIVE_BACKEND') != 'local' or 'start_webcam' not in globals()):
        raise RuntimeError('For live A/B, run and stop the earlier local live cell first.')
    if globals().get('_local_camera') is not None:
        raise RuntimeError('Stop the existing camera loop with Q before starting this experiment.')

    width = int(CAPTURE_WIDTH)
    height = max(3, round(width * 0.75))
    if image_path:
        bgr = cv2.imread(str(image_path))
        if bgr is None:
            raise ValueError(f'Cannot read image: {image_path}')
        height = max(3, round(width * bgr.shape[0] / bgr.shape[1]))
        scene = cv2.cvtColor(cv2.resize(bgr, (width, height)), cv2.COLOR_BGR2RGB)
        frames = [np.ascontiguousarray(np.roll(scene, shift, axis=1)) for shift in (0, 3, 7, 11)]
        source = 'photo and shifted variants (compute-only)'
    else:
        y, x = np.mgrid[:height, :width]
        scene = np.stack((x % 256, y % 256, (x + y) % 256), -1).astype(np.uint8)
        frames = [np.ascontiguousarray(np.roll(scene, shift, axis=1)) for shift in (0, 3, 7, 11)]
        source = 'synthetic gradients (compute-only; real-scene quality still needs inspection)'

    # Freeze only the experiment's preprocessing configuration, never mutate processor/model.
    import types
    prep_globals = dict(prepare_depth_inputs.__globals__)
    prep_globals['processor'] = copy.deepcopy(processor)
    prepare = types.FunctionType(prepare_depth_inputs.__code__, prep_globals,
                                 prepare_depth_inputs.__name__, prepare_depth_inputs.__defaults__)
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    stream = torch.cuda.current_stream(device)
    device_id = device.index if device.index is not None else torch.cuda.current_device()
    sample = prepare(frames[0])['pixel_values']
    fixed_shape = tuple(sample.shape)
    run_dir = Path('models/onnx_experiments') / datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    run_dir.mkdir(parents=True, exist_ok=False)
    report = dict(source=source, gpu=torch.cuda.get_device_name(device), torch=torch.__version__,
                  cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
                  onnxruntime=ort.__version__, dtype=str(dtype), input_shape=list(fixed_shape),
                  frame_shape=list(frames[0].shape), fast_preprocess=bool(FAST_PREPROCESS))
    print(f'Experiment: {run_dir}\n{source}\nActual model input: {fixed_shape}; {dtype}')

    class ExportDepth(torch.nn.Module):
        def __init__(self, depth_model):
            super().__init__()
            self.depth_model = depth_model

        def forward(self, pixel_values):
            return self.depth_model(pixel_values=pixel_values).predicted_depth

    with torch.inference_mode():
        wrapper = ExportDepth(model).eval()
        output_shape = tuple(wrapper(sample).shape)
        model_path = run_dir / 'depth.onnx'
        print('Exporting fixed-shape depth model (one-time cost, excluded from timing)...')
        torch.onnx.export(wrapper, (sample,), str(model_path), input_names=['pixel_values'],
                          output_names=['predicted_depth'], opset_version=17,
                          do_constant_folding=True, dynamic_axes=None, dynamo=False)
    onnx.checker.check_model(str(model_path))
    options = ort.SessionOptions()
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    options.enable_profiling = True  # One warmup run only; steady-state profiling is disabled below.
    options.profile_file_prefix = str(run_dir / 'provider_profile')
    cuda_options = dict(device_id=device_id, user_compute_stream=str(stream.cuda_stream),
                        cudnn_conv_algo_search='HEURISTIC', cudnn_conv_use_max_workspace='0')
    session = ort.InferenceSession(str(model_path), sess_options=options,
                                   providers=[('CUDAExecutionProvider', cuda_options), 'CPUExecutionProvider'])
    session.disable_fallback()
    if session.get_providers()[0] != 'CUDAExecutionProvider':
        raise RuntimeError('ONNX CUDA initialization failed; refusing to benchmark a CPU fallback.')
    output_dtype = {'tensor(float16)': torch.float16, 'tensor(float)': torch.float32}[session.get_outputs()[0].type]
    bound_output = torch.empty(output_shape, device=device, dtype=output_dtype)
    binding = session.io_binding()
    binding.bind_output('predicted_depth', 'cuda', device_id,
                        np.float16 if output_dtype == torch.float16 else np.float32,
                        output_shape, bound_output.data_ptr())

    @torch.inference_mode()
    def predict_onnx(pixels):
        if tuple(pixels.shape) != fixed_shape:
            raise ValueError(f'Export expects {fixed_shape}, received {tuple(pixels.shape)}. '
                             'Match the camera aspect ratio or rerun with a matching ONNX_IMAGE_PATH.')
        if pixels.dtype != dtype or pixels.device != device or not pixels.is_contiguous():
            raise ValueError('ONNX input must be contiguous and match the export device/dtype.')
        if torch.cuda.current_stream(device) != stream:
            raise RuntimeError('Run ONNX on the CUDA stream used when creating this experiment.')
        binding.bind_input('pixel_values', 'cuda', device_id,
                           np.float16 if dtype == torch.float16 else np.float32,
                           fixed_shape, pixels.data_ptr())
        session.run_with_iobinding(binding)
        return bound_output  # Reused buffer; consume on the same stream before the next call.

    predict_onnx(sample)
    torch.cuda.synchronize(device)
    profile_path = session.end_profiling()
    events = json.loads(Path(profile_path).read_text())
    provider_counts = {}
    for event in events:
        provider = event.get('args', {}).get('provider')
        if provider:
            provider_counts[provider] = provider_counts.get(provider, 0) + 1
    if not provider_counts.get('CUDAExecutionProvider'):
        raise RuntimeError('Profile contains no CUDA kernels; stopping the experiment.')
    report['profile_kernel_events'] = provider_counts
    print(f'Verified executed providers: {provider_counts} (CPU shape operations may remain).')

    @torch.inference_mode()
    def infer(frame, backend):
        pixels = prepare(frame)['pixel_values']
        prediction = model(pixel_values=pixels).predicted_depth if backend == 'pytorch' else predict_onnx(pixels)
        return F.interpolate(prediction[:, None].float(), size=frame.shape[:2],
                             mode='bilinear', align_corners=False)[0, 0]

    # Same normalization/smoothing for both; independent state avoids contaminating the baseline.
    def geometry(raw, mapper, estimator):
        if isinstance(mapper, DeviceRelativeGeometryDepth):
            z = mapper(raw, temporal=True, validate=False)
        else:
            z = mapper(raw.cpu().numpy(), temporal=True)
        z = torch.as_tensor(z, device=device, dtype=torch.float32)
        normals, points, valid = estimator.compute_normals_and_coords(z, validate=False)
        return z, normals, points, valid

    quality = []
    pictures = None
    with torch.inference_mode():
        for frame in frames:
            baseline, candidate = infer(frame, 'pytorch'), infer(frame, 'onnx')
            if not torch.isfinite(candidate).all() or not torch.isfinite(baseline).all():
                raise RuntimeError('Non-finite depth detected; ONNX experiment stopped, baseline unchanged.')
            scale = (torch.quantile(baseline.flatten(), .98) - torch.quantile(baseline.flatten(), .02)).clamp_min(1e-6)
            nrmse = float(torch.mean((baseline-candidate)**2).sqrt() / scale)
            estimator = ScharrNormalEstimator(*frame.shape[:2], hfov_deg=ASSUMED_HFOV_DEG, device=device)
            a = geometry(baseline, RelativeGeometryDepth(), estimator)
            b = geometry(candidate, RelativeGeometryDepth(), estimator)
            mask = a[3] & b[3]
            if not mask.any():
                raise RuntimeError('No valid geometry for ONNX quality comparison.')
            angles = torch.rad2deg(torch.acos((a[1]*b[1]).sum(-1).clamp(-1, 1)))[mask]
            ra = relight_rgb(frame, a[1], a[2], a[3])
            rb = relight_rgb(frame, b[1], b[2], b[3])
            mae = float(np.abs(ra.astype(np.float32)-rb.astype(np.float32)).mean())
            quality.append(dict(depth_nrmse=nrmse, normal_mean_deg=float(angles.mean()),
                                normal_p95_deg=float(torch.quantile(angles, .95)), relit_mae_255=mae))
            if pictures is None:
                pictures = (frame, ra, rb)
    report['quality'] = quality
    print('Quality:', {key: round(max(q[key] for q in quality), 5) for key in quality[0]})
    # Experiment guardrails, not a guarantee of real-scene visual equivalence.
    quality_ok = all(q['depth_nrmse'] < .02 and q['normal_mean_deg'] < 3 and q['relit_mae_255'] < 3 for q in quality)
    report['quality_guardrails_passed'] = quality_ok
    cv2.imwrite(str(run_dir / 'quality_rgb_pytorch_onnx.jpg'),
                cv2.cvtColor(np.concatenate(pictures, axis=1), cv2.COLOR_RGB2BGR))
    if not quality_ok:
        (run_dir / 'results.json').write_text(json.dumps(report, indent=2))
        raise RuntimeError(f'ONNX quality guardrails failed. Inspect {run_dir}; original backend is unchanged.')

    # Fix the geometry choice across backends for a fair end-to-end compute comparison.
    choice = globals().get('GEOMETRY_BACKEND', 'auto')
    estimator = ScharrNormalEstimator(height, width, hfov_deg=ASSUMED_HFOV_DEG, device=device)
    if choice == 'auto':
        if 'select_geometry_mapper' in globals():
            _, choice = select_geometry_mapper(infer(frames[0], 'pytorch'), estimator)
        else:
            choice = 'cpu'
    factory = {'cpu': RelativeGeometryDepth, 'gpu': DeviceRelativeGeometryDepth}[choice]
    report['geometry_backend'] = choice

    def paired_times(functions):
        samples = {name: [] for name in functions}
        with torch.inference_mode():
            for i in range(10):
                for fn in functions.values():
                    fn(frames[i % len(frames)])
            torch.cuda.synchronize(device)
            for i in range(count):
                order = list(functions) if i % 2 == 0 else list(reversed(functions))
                for name in order:
                    torch.cuda.synchronize(device)
                    start = time.perf_counter()
                    functions[name](frames[i % len(frames)])
                    torch.cuda.synchronize(device)
                    samples[name].append((time.perf_counter()-start)*1000)
        return {name: dict(mean_ms=float(np.mean(values)), p50_ms=float(np.median(values)),
                           p95_ms=float(np.percentile(values, 95)), compute_fps=1000/float(np.mean(values)))
                for name, values in samples.items()}

    report['depth_path'] = paired_times({name: (lambda frame, n=name: infer(frame, n))
                                         for name in ('pytorch', 'onnx')})
    mappers = {name: factory() for name in ('pytorch', 'onnx')}

    def render_compute(frame, backend):
        raw = infer(frame, backend)
        z, normals, points, valid = geometry(raw, mappers[backend], estimator)
        return relight_rgb(frame, normals, points, valid, return_tensor=True).cpu().numpy()

    report['render_compute'] = paired_times({name: (lambda frame, n=name: render_compute(frame, n))
                                             for name in ('pytorch', 'onnx')})
    for stage in ('depth_path', 'render_compute'):
        a, b = report[stage]['pytorch'], report[stage]['onnx']
        print(f'{stage}: PyTorch {a["mean_ms"]:.2f} ms ({a["compute_fps"]:.2f}/s), '
              f'ONNX {b["mean_ms"]:.2f} ms ({b["compute_fps"]:.2f}/s); '
              f'{a["mean_ms"]/b["mean_ms"]:.2f}x speedup')
    print('Compute rates exclude camera, hand tracking, UI and display. They are not live FPS.')
    (run_dir / 'results.json').write_text(json.dumps(report, indent=2))

    if live:
        # Explicit backend argument: never replace infer_relative_inverse or run_live.
        report['live'] = {}
        for backend in ('pytorch', 'onnx'):
            durations, hand_modes = [], set()
            mapper, live_estimator = factory(), None
            print(f'Live {backend.upper()}: {live_frames} measured frames after 10 warmups. '
                  'Same settings; separate camera runs. Q/Esc cancels the comparison.')
            cancelled = False
            try:
                if not start_webcam():
                    break
                with torch.inference_mode():
                    for i in range(live_frames + 10):
                        start = time.perf_counter()
                        packet = get_frame()  # Includes actual same-frame MediaPipe tracking.
                        if packet is None:
                            cancelled = True
                            break
                        frame, light, hand = packet
                        h, w = frame.shape[:2]
                        if live_estimator is None:
                            if tuple(prepare(frame)['pixel_values'].shape) != fixed_shape:
                                raise ValueError('Camera aspect ratio differs from export; use a matching scene photo to export.')
                            live_estimator = ScharrNormalEstimator(h, w, hfov_deg=ASSUMED_HFOV_DEG, device=device)
                        elif (live_estimator.H, live_estimator.W) != (h, w):
                            raise RuntimeError('Camera resolution changed; rerun the experiment.')
                        raw = infer(frame, backend)
                        z, normals, points, valid = geometry(raw, mapper, live_estimator)
                        relit = relight_rgb(frame, normals, points, valid, light, return_tensor=True)
                        if SHOW_DIAGNOSTICS:
                            nrgb = torch.where(valid[..., None], ((normals+1)*127.5).round().clamp(0,255), 0).to(torch.uint8)
                            picture = np.asarray(live_panel(frame, z.cpu().numpy(), nrgb.cpu().numpy(), relit.cpu().numpy())).copy()
                        else:
                            picture = relit.cpu().numpy().copy()
                        annotate_local_hand(picture, hand, h, w)
                        rate = f'{len(durations)/sum(durations):.1f} FPS' if durations else 'warmup'
                        cv2.putText(picture, f'{backend.upper()} | {rate} | Q: cancel', (8,18),
                                    cv2.FONT_HERSHEY_SIMPLEX, .45, (0,255,160), 1, cv2.LINE_AA)
                        if not show_local_frame(picture):
                            cancelled = True
                            break
                        if i >= 10:
                            durations.append(time.perf_counter()-start)
                            hand_modes.add(hand['mode'])
            except KeyboardInterrupt:
                cancelled = True
            finally:
                stop_webcam()
                if durations:
                    report['live'][backend] = dict(frames=len(durations), fps=len(durations)/sum(durations),
                                                   p95_ms=float(np.percentile(durations,95)*1000),
                                                   hand_modes=sorted(hand_modes), cancelled=cancelled)
                    print(backend, report['live'][backend])
                (run_dir / 'results.json').write_text(json.dumps(report, indent=2))
            if cancelled:
                break
        if all(name in report['live'] for name in ('pytorch', 'onnx')):
            a, b = report['live']['pytorch'], report['live']['onnx']
            if (a['frames'] == b['frames'] == live_frames and not a['cancelled'] and not b['cancelled']
                    and a['hand_modes'] == b['hand_modes']):
                print(f'Live FPS change: {(b["fps"]/a["fps"]-1)*100:+.1f}%. '
                      'Separate camera runs; repeat in reverse order to check thermal/scene effects.')
            else:
                print('Live comparison incomplete or hand modes differed; do not claim an FPS improvement.')
    print(f'Saved experiment only: {run_dir / "results.json"}')
    return report


try:
    import onnx  # noqa: F401
    import onnxruntime  # noqa: F401
except ImportError:
    print('Skipping optional ONNX experiment: install the pinned ONNX dependencies to enable it.')
else:
    onnx_experiment_results = run_onnx_experiment(
        live=ONNX_LIVE_COMPARE, image_path=ONNX_IMAGE_PATH,
        count=ONNX_BENCHMARK_FRAMES, live_frames=ONNX_LIVE_FRAMES,
    )


Skipping optional ONNX experiment: install the pinned ONNX dependencies to enable it.
